# Recalage de la classification Lidar depuis les couches vecteur BD TOPO

Ce notebook vérifie que `processing.lidar.classify.classify_from_vectors` (PDAL `filters.overlay`) recale la classification des points selon les polygones bâtiment/hydro/végétation déjà téléchargés (BD TOPO), en complément de la classification IGN déjà présente dans les dalles Lidar HD.

**Note sur la performance** : `filters.overlay` teste chaque point contre chaque couche vecteur. Sur le nuage complet (29,5M points), un premier essai avec 3 couches a été interrompu après plus de 20 minutes sans terminer (extrapolation : plusieurs heures). Ce n'est pas adapté à un run pipeline complet sans découpage préalable — la classification doit donc être appliquée **après** le découpage à l'emprise de la commune (qui réduit drastiquement le nombre de points), jamais sur le nuage brut complet. Ce notebook illustre la fonction sur une petite zone de test (120×120 m, ~330 000 points) pour rester rapide.

In [ ]:
import json
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

import geopandas as gpd
import numpy as np
import pdal
from shapely.geometry import box

from processing.lidar.clip import clip_las_to_boundary
from processing.lidar.classify import classify_from_vectors

merged_path = repo_root / "data" / "raw" / "raster" / "lidar" / "nuage_points.las"
bd_topo_dir = repo_root / "data" / "raw" / "vector" / "bd_topo"

## 1. Sélection d'une petite zone de test autour d'un bâtiment réel

In [ ]:
batiment = gpd.read_file(bd_topo_dir / "batiment.gpkg")
tile_box = box(886000, 6499000, 887000, 6500000)
bat_in_tile = batiment[batiment.intersects(tile_box)].reset_index(drop=True)

b0 = bat_in_tile.iloc[10].geometry
cx, cy = b0.centroid.x, b0.centroid.y
print(f"Batiment choisi : centroide=({cx:.1f},{cy:.1f}), bounds={b0.bounds}")

small_box_geom = box(cx - 60, cy - 60, cx + 60, cy + 60)
small_boundary = gpd.GeoDataFrame({"id": [1]}, geometry=[small_box_geom], crs="EPSG:2154")

clipped_path = repo_root / "data" / "processed" / "raster" / "lidar" / "_test_zone.las"
clip_las_to_boundary(merged_path, small_boundary, buffer_distance=0, output_path=clipped_path)

pipeline = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(clipped_path)}]))
count_before = pipeline.execute()
arr_before = pipeline.arrays[0]
print("Points dans la zone de test :", count_before)

## 2. Recalage de la classification (végétation, bâtiment)

In [ ]:
layers = {
    "vegetation": (bd_topo_dir / "zone_de_vegetation.gpkg", 5),
    "batiment": (bd_topo_dir / "batiment.gpkg", 6),
}

output_path = repo_root / "data" / "processed" / "raster" / "lidar" / "_test_zone_classifiee.las"
result = classify_from_vectors(clipped_path, output_path, layers)

pipeline2 = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(result)}]))
pipeline2.execute()
arr_after = pipeline2.arrays[0]

classes_before, counts_before = np.unique(arr_before["Classification"], return_counts=True)
classes_after, counts_after = np.unique(arr_after["Classification"], return_counts=True)
print("Repartition AVANT (IGN) :", list(zip(classes_before.tolist(), counts_before.tolist())))
print("Repartition APRES (vecteur) :", list(zip(classes_after.tolist(), counts_after.tolist())))

## 3. Vérification ciblée : points dans l'emprise exacte du bâtiment

In [ ]:
mask_before = (
    (arr_before["X"] > b0.bounds[0]) & (arr_before["X"] < b0.bounds[2]) &
    (arr_before["Y"] > b0.bounds[1]) & (arr_before["Y"] < b0.bounds[3])
)
mask_after = (
    (arr_after["X"] > b0.bounds[0]) & (arr_after["X"] < b0.bounds[2]) &
    (arr_after["Y"] > b0.bounds[1]) & (arr_after["Y"] < b0.bounds[3])
)

print("Points dans l'emprise du batiment :", mask_before.sum())
print("Classes AVANT recalage :", list(zip(*np.unique(arr_before["Classification"][mask_before], return_counts=True))))
print("Classes APRES recalage :", list(zip(*np.unique(arr_after["Classification"][mask_after], return_counts=True))))

n_bati_before = (arr_before["Classification"][mask_before] == 6).sum()
n_bati_after = (arr_after["Classification"][mask_after] == 6).sum()
print(f"\nPoints classes 'batiment' (6) dans l'emprise : {n_bati_before} -> {n_bati_after} ({100 * n_bati_after / mask_after.sum():.1f}% du total)")

## 4. Aperçu avant/après

In [ ]:
import matplotlib.pyplot as plt

colors_map = {1: "gray", 2: "saddlebrown", 3: "yellowgreen", 4: "forestgreen", 5: "darkgreen", 6: "crimson", 9: "blue", 64: "gray", 67: "gray"}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, arr_plot, title in [(axes[0], arr_before, "Classification IGN (avant)"), (axes[1], arr_after, "Classification recalee (apres)")]:
    for cls in np.unique(arr_plot["Classification"]):
        m = arr_plot["Classification"] == cls
        ax.scatter(arr_plot["X"][m], arr_plot["Y"][m], s=0.5, color=colors_map.get(int(cls), "black"), label=f"classe {cls}")
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend(markerscale=15, fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

clipped_path.unlink()
output_path.unlink()